In [ ]:
from haystack.document_stores import FAISSDocumentStore
from haystack.nodes import EmbeddingRetriever, PDFToTextConverter, TextConverter, DocxToTextConverter
from haystack.pipelines import DocumentSearchPipeline
from haystack.utils import clean_wiki_text, convert_files_to_docs, fetch_archive_from_http

# --- 1. Initialize document store (FAISS) ---
document_store = FAISSDocumentStore(faiss_index_factory_str="Flat")

# --- 2. Convert and load documents ---
docs = convert_files_to_docs(dir_path="data/", clean_func=clean_wiki_text, split_paragraphs=True)

# --- 3. Write docs to document store ---
document_store.write_documents(docs)

# --- 4. Embed documents with retriever ---
retriever = EmbeddingRetriever(document_store=document_store,
                               embedding_model="sentence-transformers/all-MiniLM-L6-v2")

# Generate embeddings for docs
document_store.update_embeddings(retriever)

# --- 5. Create pipeline and run query ---
pipe = DocumentSearchPipeline(retriever)

query = "email about project budget"
result = pipe.run(query=query, params={"Retriever": {"top_k": 3}})

for doc in result["documents"]:
    print(doc.meta["name"], "\n", doc.content[:300], "\n---")
